In [86]:
import ast
import re
import unicodedata
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

try:
    from rapidfuzz import fuzz
except Exception:  # fallback
    from difflib import SequenceMatcher

    class _Fuzz:
        @staticmethod
        def partial_ratio(a, b):
            return int(100 * SequenceMatcher(None, a, b).ratio())

    fuzz = _Fuzz()

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    SentenceTransformer = None


DEFAULT_SPLIT_MARKERS = [
    "отдельно",
    "отдельные услуги",
    "оказываем услуги",
    "услуги",
    "выполняем",
    "мастер",
    "бригада",
    "выезд",
    "профессионально",
    "специалист",
    "специалисты",
    "работаем",
    "монтаж",
    "установка",
    "замена",
    "ремонт",
    "подключение",
    "демонтаж",
    "сборка",
]

DEFAULT_COMPLEXITY_MARKERS = [
    "под ключ",
    "комплексный ремонт",
    "ремонт квартир",
    "ремонт дома",
    "ремонт домов",
    "капитальный ремонт",
    "косметический ремонт",
    "отделка",
    "черновая отделка",
    "чистовая отделка",
    "в составе ремонта",
    "весь спектр",
    "полный цикл",
    "все виды работ",
    "с нуля",
]


In [87]:
def _is_na(x: Any) -> bool:
    return x is None or (isinstance(x, float) and np.isnan(x))

In [88]:
def normalize_text(text: Any) -> str:
    if _is_na(text):
        return ""
    text = str(text).lower().replace("ё", "е")
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^0-9a-zа-я]+", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [89]:
def ensure_list(x: Any) -> List[Any]:
    if _is_na(x):
        return []
    if isinstance(x, (list, tuple, set, np.ndarray)):
        return list(x)
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple, set)):
                return list(parsed)
        except Exception:
            pass
        return [p.strip() for p in s.split(";") if p.strip()]
    return [x]

In [90]:
def ensure_int_list(x: Any) -> List[int]:
    out = []
    seen = set()
    for v in ensure_list(x):
        try:
            iv = int(v)
        except Exception:
            continue
        if iv not in seen:
            out.append(iv)
            seen.add(iv)
    return out

In [91]:
def parse_keyphrases(x: Any) -> List[str]:
    if _is_na(x):
        return []
    if isinstance(x, (list, tuple, set, np.ndarray)):
        raw = list(x)
    else:
        s = str(x).strip()
        if not s:
            return []
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple, set)):
                raw = list(parsed)
            else:
                raw = re.split(r"[;|]", s)
        except Exception:
            raw = re.split(r"[;|]", s)

    out = []
    seen = set()
    for p in raw:
        p = normalize_text(p)
        if p and p not in seen:
            out.append(p)
            seen.add(p)
    return out

In [92]:
def dedupe_preserve_order(seq: Iterable[Any]) -> List[Any]:
    out = []
    seen = set()
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out

In [93]:
def _best_fuzz_score(text: str, phrases: Sequence[str], fallback: Optional[str] = None) -> float:
    candidates = list(phrases)
    if not candidates and fallback:
        candidates = [fallback]
    if not candidates or not text:
        return 0.0

    scores = []
    for p in candidates:
        if not p:
            continue
        scores.append(fuzz.partial_ratio(text, p) / 100.0)
    return float(max(scores)) if scores else 0.0

In [94]:
def _count_occurrences(text: str, phrases: Sequence[str]) -> int:
    if not text or not phrases:
        return 0
    total = 0
    for p in phrases:
        if p:
            total += text.count(p)
    return total

In [95]:
def _find_first_phrase_span(
    text: str,
    phrases: Sequence[str],
    radius: int = 35,
) -> Optional[Tuple[int, int, int, int, str]]:
    if not text or not phrases:
        return None

    best = None
    for p in phrases:
        idx = text.find(p)
        if idx == -1:
            continue
        end = idx + len(p)
        if best is None or idx < best[0] or (idx == best[0] and len(p) > len(best[4])):
            best = (idx, end, max(0, idx - radius), min(len(text), end + radius), p)

    if best is None:
        return None
    return best

In [96]:
def l2_normalize(mat: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(mat, axis=1, keepdims=True)
    norm = np.clip(norm, 1e-12, None)
    return mat / norm


def cosine_sim_1d(a: np.ndarray, b: np.ndarray) -> float:
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom <= 1e-12:
        return 0.0
    return float(np.dot(a, b) / denom)


def to_text_list(x):
    """
    candidate_mc_doc_norm может быть строкой или списком строк.
    """
    if x is None:
        return []
    if isinstance(x, list):
        return [str(t) for t in x if str(t).strip()]
    if isinstance(x, tuple):
        return [str(t) for t in x if str(t).strip()]
    s = str(x).strip()
    return [s] if s else []


def mean_embedding(model, texts):
    """
    texts: list[str]
    returns: np.ndarray shape (dim,)
    """
    if not texts:
        return None

    emb = model.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    if emb.ndim == 1:
        return emb
    return emb.mean(axis=0)

In [97]:
def _ratio(a: int, b: int) -> float:
    return float(a / b) if b else 0.0

In [98]:
@dataclass
class RepairDatasetBuilder:
    split_markers: Sequence[str] = tuple(DEFAULT_SPLIT_MARKERS)
    complexity_markers: Sequence[str] = tuple(DEFAULT_COMPLEXITY_MARKERS)
    span_radius: int = 35
    tfidf_ngram_range: Tuple[int, int] = (1, 2)
    tfidf_min_df: int = 1

    embedding_model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

    def __post_init__(self) -> None:
        self.split_markers_ = [normalize_text(x) for x in self.split_markers if normalize_text(x)]
        self.complexity_markers_ = [normalize_text(x) for x in self.complexity_markers if normalize_text(x)]
        self.vectorizer_: Optional[TfidfVectorizer] = None
        self.mc_meta_: Optional[pd.DataFrame] = None
        self.apriori_map_: Dict[int, float] = {}
        self.global_prior_: float = 0.0
        self.embedder_ = None

    def _init_embedder(self):
        if self.embedder_ is not None:
            return self.embedder_
        if SentenceTransformer is None:
            raise ImportError(
                "sentence-transformers не установлен. "
                "Установи пакет: pip install sentence-transformers"
            )
        self.embedder_ = SentenceTransformer(self.embedding_model_name)
        return self.embedder_
    
    def prepare_mc_meta(self, mc_meta_df: pd.DataFrame) -> pd.DataFrame:
        mc = mc_meta_df.copy()

        if "mcId" not in mc.columns:
            raise ValueError("mc_meta_df must contain column 'mcId'")

        mc["mcId"] = mc["mcId"].astype(int)
        mc["mcTitle"] = mc.get("mcTitle", "").fillna("").astype(str)
        mc["description"] = mc.get("description", "").fillna("").astype(str)
        mc["keyPhrases"] = mc.get("keyPhrases", "").apply(parse_keyphrases)

        mc["mcTitle_norm"] = mc["mcTitle"].map(normalize_text)
        mc["description_norm"] = mc["description"].map(normalize_text)

        mc["mc_doc_norm"] = mc.apply(
            lambda r: normalize_text(
                " ".join(
                    [
                        r["mcTitle"],
                        r["description"],
                        " ".join(r["keyPhrases"]),
                    ]
                )
            ),
            axis=1,
        )

        return mc.set_index("mcId", drop=False)

    def _build_pairs(self, raw_df: pd.DataFrame, with_label: bool = True) -> pd.DataFrame:
        if self.mc_meta_ is None:
            raise RuntimeError("Call fit() first.")

        rows = []

        for row in raw_df.itertuples(index=False):
            item = row._asdict()
            desc_raw = item.get("description", "")
            desc_norm = normalize_text(desc_raw)

            detected = dedupe_preserve_order(ensure_int_list(item.get("targetDetectedMcIds")))
            split_set = set(ensure_int_list(item.get("targetSplitMcIds")))
            item_id = item.get("itemId", None)

            for rank, cand_id in enumerate(detected):
                meta = self.mc_meta_.loc[cand_id] if cand_id in self.mc_meta_.index else None

                cand_title = "" if meta is None else str(meta["mcTitle"])
                cand_title_norm = "" if meta is None else str(meta["mcTitle_norm"])
                cand_desc_norm = "" if meta is None else str(meta["description_norm"])
                cand_doc_norm = "" if meta is None else str(meta["mc_doc_norm"])
                cand_phrases = [] if meta is None else list(meta["keyPhrases"])

                rec = {
                    "itemId": item_id,
                    "description": str(desc_raw) if not _is_na(desc_raw) else "",
                    "description_norm": desc_norm,
                    "candidate_mcId": int(cand_id),
                    "candidate_mcTitle": cand_title,
                    "candidate_mcTitle_norm": cand_title_norm,
                    "candidate_description_norm": cand_desc_norm,
                    "candidate_keyphrases": cand_phrases,
                    "candidate_keyphrases_norm": cand_phrases,
                    "candidate_mc_doc_norm": cand_doc_norm,
                    "candidate_rank": rank,
                }

                if with_label:
                    rec["label"] = int(cand_id in split_set)

                rows.append(rec)

        out = pd.DataFrame(rows)
        if out.empty:
            return out
        if with_label and "label" not in out.columns:
            out["label"] = 0
        return out

    def fit(self, raw_df: pd.DataFrame, mc_meta_df: pd.DataFrame) -> "RepairDatasetBuilder":
        self.mc_meta_ = self.prepare_mc_meta(mc_meta_df)
        flat = self._build_pairs(raw_df, with_label=True)

        corpus = pd.concat(
            [
                flat["description_norm"],
                flat["candidate_mc_doc_norm"],
            ],
            ignore_index=True,
        ).fillna("").astype(str).tolist()

        self.vectorizer_ = TfidfVectorizer(
            ngram_range=self.tfidf_ngram_range,
            min_df=self.tfidf_min_df,
            lowercase=False,
            norm="l2",
            sublinear_tf=True,
        )
        self.vectorizer_.fit(corpus)

        self.apriori_map_ = flat.groupby("candidate_mcId")["label"].mean().to_dict()
        self.global_prior_ = float(flat["label"].mean()) if len(flat) else 0.0
        return self

    def _row_features(
        self,
        desc_norm: str,
        cand_title_norm: str,
        cand_phrases: Sequence[str],
    ) -> Dict[str, float]:
        text = desc_norm
        words = text.split()

        # text_length — здесь это число токенов после нормализации
        text_length = float(len(words))

        first_span = _find_first_phrase_span(text, cand_phrases, radius=self.span_radius)

        key_phrases_matches = 0.0
        has_span = 0.0
        span_normalized_position = 0.0
        span_ratio_split_complex = 0.0

        if first_span is not None:
            first_start, first_end, span_start, span_end, _ = first_span
            has_span = 1.0
            key_phrases_matches = float(sum(1 for p in cand_phrases if p and p in text))
            span_normalized_position = _ratio(first_start, max(1, len(text)))

            span_text = text[span_start:span_end]
            split_count_span = _count_occurrences(span_text, self.split_markers_)
            complex_count_span = _count_occurrences(span_text, self.complexity_markers_)
            span_ratio_split_complex = _ratio(
                split_count_span,
                split_count_span + complex_count_span,
            )

        split_count_global = _count_occurrences(text, self.split_markers_)
        complex_count_global = _count_occurrences(text, self.complexity_markers_)
        global_ratio_split_complex = _ratio(
            split_count_global,
            split_count_global + complex_count_global,
        )

        fuzz_score = _best_fuzz_score(text, cand_phrases, fallback=cand_title_norm)
        candidate_title_match = float(bool(cand_title_norm and cand_title_norm in text))
        candidate_title_fuzz = _best_fuzz_score(text, [cand_title_norm] if cand_title_norm else [], fallback=None)

        return {
            "text_length": text_length,
            "fuzz_score": float(fuzz_score),
            "split_markers_number": float(split_count_global),
            "complexity_markers_number": float(complex_count_global),
            "key_phrases_matches": float(key_phrases_matches),
            "has_span": float(has_span),
            "span_normalized_position": float(span_normalized_position),
            "span_ratio_split_complex": float(span_ratio_split_complex),
            "global_ratio_split_complex": float(global_ratio_split_complex),
            # extra features
            "candidate_title_match": candidate_title_match,
            "candidate_title_fuzz": float(candidate_title_fuzz),
        }

    def transform(self, raw_df: pd.DataFrame) -> pd.DataFrame:
        if self.vectorizer_ is None or self.mc_meta_ is None:
            raise RuntimeError("Call fit() first.")

        flat = self._build_pairs(raw_df, with_label=("targetSplitMcIds" in raw_df.columns))

        # --- TF-IDF признаки ---
        desc_vec = self.vectorizer_.transform(flat["description_norm"].fillna("").astype(str).tolist())

        # candidate_mc_doc_norm здесь может быть строкой или списком строк
        candidate_docs = flat["candidate_mc_doc_norm"].apply(to_text_list).tolist()

        # tf-idf_score: считаем по первому/объединённому тексту кандидата
        # Если у тебя там всегда строка, это будет работать как раньше.
        candidate_doc_texts = [
            " ".join(docs).strip() if docs else ""
            for docs in candidate_docs
        ]
        cand_vec = self.vectorizer_.transform(candidate_doc_texts)

        mc_bin = cand_vec.copy()
        if mc_bin.nnz:
            mc_bin.data = np.ones_like(mc_bin.data)

        tfidf_score = np.asarray(desc_vec.multiply(mc_bin).sum(axis=1)).ravel()

        # --- Sentence embeddings + cosine similarity ---
        embedder = self._init_embedder()

        # описание — поштучно
        desc_texts = flat["description_norm"].fillna("").astype(str).tolist()
        desc_emb = embedder.encode(
            desc_texts,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )

        # кандидат — средний эмбеддинг по списку candidate_mc_doc_norm
        cand_emb_list = []
        for docs in candidate_docs:
            if not docs:
                cand_emb_list.append(None)
                continue
            emb = embedder.encode(
                docs,
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=False,
            )
            if emb.ndim == 1:
                cand_emb = emb
            else:
                cand_emb = emb.mean(axis=0)
                cand_emb = cand_emb / max(np.linalg.norm(cand_emb), 1e-12)
            cand_emb_list.append(cand_emb)

        cos_sim_score = []
        for d_emb, c_emb in zip(desc_emb, cand_emb_list):
            if c_emb is None:
                cos_sim_score.append(0.0)
            else:
                cos_sim_score.append(cosine_sim_1d(d_emb, c_emb))

        feat_rows = []
        for desc_norm, cand_title_norm, cand_phrases in zip(
            flat["description_norm"].fillna("").astype(str),
            flat["candidate_mcTitle_norm"].fillna("").astype(str),
            flat["candidate_keyphrases_norm"].apply(list),
        ):
            feat_rows.append(self._row_features(desc_norm, cand_title_norm, cand_phrases))

        feat = pd.DataFrame(feat_rows)
        feat["tf-idf_score"] = tfidf_score
        feat["cos_sim_score"] = np.asarray(cos_sim_score, dtype=float)

        apr = flat["candidate_mcId"].map(self.apriori_map_).fillna(self.global_prior_).astype(float)
        feat["apriory_pair_split_rate"] = apr.values

        return pd.concat([flat.reset_index(drop=True), feat.reset_index(drop=True)], axis=1)

    def fit_transform(self, raw_df: pd.DataFrame, mc_meta_df: pd.DataFrame) -> pd.DataFrame:
        self.fit(raw_df, mc_meta_df)
        return self.transform(raw_df)

builder = RepairDatasetBuilder()


In [99]:
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/rnc_dataset_markup.csv", sep=";").drop(["sourceMcId", "sourceMcTitle", "shouldSplit", "caseType", "split"], axis=1)
mc_meta_df = pd.read_csv("../data/csv_files/rnc_mic_key_phrases.csv")

raw_train_df, raw_valid_df = train_test_split(
    df,
    test_size=0.2,      # 20% в валидацию
    random_state=42     # фиксируем случайность для воспроизводимости
)


train_pairs_df = builder.fit_transform(raw_train_df, mc_meta_df)
valid_pairs_df = builder.transform(raw_valid_df)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [100]:
train_pairs_df

,itemId,description,description_norm,candidate_mcId,candidate_mcTitle,candidate_mcTitle_norm,candidate_description_norm,candidate_keyphrases,candidate_keyphrases_norm,candidate_mc_doc_norm,...,key_phrases_matches,has_span,span_normalized_position,span_ratio_split_complex,global_ratio_split_complex,candidate_title_match,candidate_title_fuzz,tf-idf_score,cos_sim_score,apriory_pair_split_rate
0,1001787,"Уважаемые дамы и господа, предлагаем услуги ре...",уважаемые дамы и господа предлагаем услуги рем...,102,Сантехника,сантехника,сантехнические работы,"[сантехника, сантехнические работы, услуги сан...","[сантехника, сантехнические работы, услуги сан...",сантехника сантехнические работы сантехника са...,...,1.0,1.0,0.087719,0.0,0.636364,1.0,1.000000,0.060844,0.425948,0.267509
1,1001787,"Уважаемые дамы и господа, предлагаем услуги ре...",уважаемые дамы и господа предлагаем услуги рем...,103,Электрика,электрика,электромонтажные работы,"[электрика, электромонтажные работы, услуги эл...","[электрика, электромонтажные работы, услуги эл...",электрика электромонтажные работы электрика эл...,...,1.0,1.0,0.082403,0.0,0.636364,1.0,1.000000,0.136237,0.461143,0.286357
2,1001787,"Уважаемые дамы и господа, предлагаем услуги ре...",уважаемые дамы и господа предлагаем услуги рем...,104,Натяжные потолки,натяжные потолки,монтаж и обслуживание натяжных потолков,"[натяжные потолки, монтаж натяжных потолков, у...","[натяжные потолки, монтаж натяжных потолков, у...",натяжные потолки монтаж и обслуживание натяжны...,...,1.0,1.0,0.056885,0.0,0.636364,1.0,1.000000,0.161456,0.470440,0.311037
3,1001787,"Уважаемые дамы и господа, предлагаем услуги ре...",уважаемые дамы и господа предлагаем услуги рем...,108,Штукатурные работы,штукатурные работы,штукатурка и выравнивание поверхностей,"[штукатурные работы, штукатурка стен, штукатур...","[штукатурные работы, штукатурка стен, штукатур...",штукатурные работы штукатурка и выравнивание п...,...,0.0,0.0,0.000000,0.0,0.636364,0.0,0.611111,0.122969,0.472018,0.267023
4,1000769,Произведем монтаж натяжного потолка любой слож...,произведем монтаж натяжного потолка любой слож...,104,Натяжные потолки,натяжные потолки,монтаж и обслуживание натяжных потолков,"[натяжные потолки, монтаж натяжных потолков, у...","[натяжные потолки, монтаж натяжных потолков, у...",натяжные потолки монтаж и обслуживание натяжны...,...,0.0,0.0,0.000000,0.0,1.000000,0.0,0.812500,0.292350,0.494658,0.311037
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6217,1001658,"шпаклёвка,штукатурка, сантехника, плитка,ламин...",шпаклевка штукатурка сантехника плитка ламинат...,108,Штукатурные работы,штукатурные работы,штукатурка и выравнивание поверхностей,"[штукатурные работы, штукатурка стен, штукатур...","[штукатурные работы, штукатурка стен, штукатур...",штукатурные работы штукатурка и выравнивание п...,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.611111,0.079386,0.694383,0.267023
6218,1001658,"шпаклёвка,штукатурка, сантехника, плитка,ламин...",шпаклевка штукатурка сантехника плитка ламинат...,109,Напольные покрытия,напольные покрытия,монтаж и замена напольных покрытий,"[напольные покрытия, укладка ламината, укладка...","[напольные покрытия, укладка ламината, укладка...",напольные покрытия монтаж и замена напольных п...,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.444444,0.097195,0.638115,0.302147
6219,1001314,"Сделаем качественный ремонт, вашей квартиры, о...",сделаем качественный ремонт вашей квартиры офи...,102,Сантехника,сантехника,сантехнические работы,"[сантехника, сантехнические работы, услуги сан...","[сантехника, сантехнические работы, услуги сан...",сантехника сантехнические работы сантехника са...,...,0.0,0.0,0.000000,0.0,0.727273,0.0,0.900000,0.038800,0.303365,0.267509
6220,1001314,"Сделаем качественный ремонт, вашей квартиры, о...",сделаем качественный ремонт вашей квартиры офи...,103,Электрика,электрика,электромонтажные работы,"[электрика, электромонтажные работы, услуги эл...","[электрика, электромонтажные работы, 

In [101]:
FEATURES = [
    "text_length",
    "fuzz_score",
    "tf-idf_score",
    "cos_sim_score",
    "split_markers_number",
    "complexity_markers_number",
    "key_phrases_matches",
    "has_span",
    "span_normalized_position",
    "span_ratio_split_complex",
    "global_ratio_split_complex",
    "apriory_pair_split_rate",
    "candidate_title_match",   # extra
    "candidate_title_fuzz",    # extra
]

In [102]:
X_train = train_pairs_df[FEATURES]
y_train = train_pairs_df["label"]

In [103]:
X_train

,text_length,fuzz_score,tf-idf_score,cos_sim_score,split_markers_number,complexity_markers_number,key_phrases_matches,has_span,span_normalized_position,span_ratio_split_complex,global_ratio_split_complex,apriory_pair_split_rate,candidate_title_match,candidate_title_fuzz
0,277.0,1.000000,0.060844,0.425948,7.0,4.0,1.0,1.0,0.087719,0.0,0.636364,0.267509,1.0,1.000000
1,277.0,1.000000,0.136237,0.461143,7.0,4.0,1.0,1.0,0.082403,0.0,0.636364,0.286357,1.0,1.000000
2,277.0,1.000000,0.161456,0.470440,7.0,4.0,1.0,1.0,0.056885,0.0,0.636364,0.311037,1.0,1.000000
3,277.0,0.823529,0.122969,0.472018,7.0,4.0,0.0,0.0,0.000000,0.0,0.636364,0.267023,0.0,0.611111
4,38.0,0.923077,0.292350,0.494658,1.0,0.0,0.0,0.0,0.000000,0.0,1.000000,0.311037,0.0,0.812500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6217,11.0,0.866667,0.079386,0.694383,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.267023,0.0,0.611111
6218,11.0,0.714286,0.097195,0.638115,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.302147,0.0,0.444444
6219,150.0,0.900000,0.038800,0.303365,8.0,3.0,0.0,0.0,0.000000,0.0,0.727273,0.267509,0.0,0.900000
6220,150.0,0.888889,0.077346,0.415268,8.0,3.0,0.0,0.0,0.000000,0.0,0.727273,0.286357,0.0,0.888889


In [104]:
from catboost import CatBoostClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score

In [105]:
def flatten_pair_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """
    Метрики по парам (itemId, candidate_mcId).
    В этой задаче их обычно считают как precision/recall/F1 по положительному классу.
    """
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    return {
        "precision_micro": precision,
        "recall_micro": recall,
        "f1_micro": f1,
    }

In [106]:
def item_level_shouldsplit_accuracy(valid_df: pd.DataFrame) -> float:
    """
    Accuracy по shouldSplit на уровне исходного объявления.
    Для каждого itemId:
      y_true = 1, если в item есть хотя бы одна истинная split-микрокатегория
      y_pred = 1, если модель предсказала хотя бы одну split-микрокатегорию
    """
    item_true = valid_df.groupby("itemId")["label"].max()
    item_pred = valid_df.groupby("itemId")["pred_label"].max()

    item_pred = item_pred.reindex(item_true.index, fill_value=0)
    return accuracy_score(item_true.values, item_pred.values)



In [107]:
def find_best_threshold(valid_df: pd.DataFrame, n_steps: int = 181) -> tuple[float, dict]:
    """
    Подбираем порог по micro F1 на pair-level.
    """
    thresholds = np.linspace(0.05, 0.95, n_steps)
    best_thr = 0.5
    best_metrics = None
    best_f1 = -1.0

    y_true = valid_df["label"].values
    proba = valid_df["proba"].values

    for thr in thresholds:
        pred = (proba >= thr).astype(int)
        metrics = flatten_pair_metrics(y_true, pred)
        if metrics["f1_micro"] > best_f1:
            best_f1 = metrics["f1_micro"]
            best_thr = float(thr)
            best_metrics = metrics

    return best_thr, best_metrics

In [108]:
def train_and_evaluate(train_pairs_df: pd.DataFrame, valid_size: float = 0.2, random_state: int = 42):
    """
    train_pairs_df должен содержать:
      itemId, label, и все FEATURES
    """
    required_cols = {"itemId", "label", *FEATURES}
    missing = required_cols - set(train_pairs_df.columns)
    if missing:
        raise ValueError(f"Не хватает колонок: {missing}")

    # Разделение строго по itemId, чтобы не было leakage между парами одного объявления
    gss = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=random_state)
    train_idx, valid_idx = next(gss.split(train_pairs_df, train_pairs_df["label"], groups=train_pairs_df["itemId"]))

    train_df = train_pairs_df.iloc[train_idx].copy()
    valid_df = train_pairs_df.iloc[valid_idx].copy()

    X_train = train_df[FEATURES]
    y_train = train_df["label"].astype(int)

    X_valid = valid_df[FEATURES]
    y_valid = valid_df["label"].astype(int)

    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=3000,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5.0,
        random_seed=random_state,
        auto_class_weights="Balanced",
        verbose=200,
        allow_writing_files=False,
        early_stopping_rounds=150,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
    )

    valid_df["proba"] = model.predict_proba(X_valid)[:, 1]

    best_thr, pair_metrics = find_best_threshold(valid_df)
    valid_df["pred_label"] = (valid_df["proba"] >= best_thr).astype(int)

    should_split_acc = item_level_shouldsplit_accuracy(valid_df)

    result = {
        "model": model,
        "threshold": best_thr,
        "pair_metrics": pair_metrics,
        "shouldSplit_accuracy": should_split_acc,
        "valid_df_scored": valid_df,
    }
    return result

In [109]:
result = train_and_evaluate(train_pairs_df)

print("Best threshold:", result["threshold"])
print("Pair metrics:", result["pair_metrics"])
print("shouldSplit accuracy:", result["shouldSplit_accuracy"])

model = result["model"]
threshold = result["threshold"]
print(model, threshold)

0:	test: 0.5375571	best: 0.5375571 (0)	total: 2.72ms	remaining: 8.15s
200:	test: 0.6445206	best: 0.6567676 (71)	total: 276ms	remaining: 3.85s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.6567675835
bestIteration = 71

Shrink model to first 72 iterations.
Best threshold: 0.48999999999999994
Pair metrics: {'precision_micro': 0.3820395738203957, 'recall_micro': 0.6765498652291105, 'f1_micro': 0.48832684824902717}
shouldSplit accuracy: 0.5125448028673835
CatBoostClassifier(allow_writing_files=False, auto_class_weights='Balanced', depth=6, early_stopping_rounds=150, eval_metric='AUC', iterations=3000, l2_leaf_reg=5.0, learning_rate=0.03, loss_function='Logloss', random_seed=42, verbose=200) 0.48999999999999994


In [ ]:
import json
from pathlib import Path

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

# model — это уже обученный CatBoostClassifier
model_path = ARTIFACT_DIR / "catboost_repair_split_2.cbm"
threshold_path = ARTIFACT_DIR / "catboost_repair_split_threshold_2.json"

model.save_model(str(model_path))

with open(threshold_path, "w", encoding="utf-8") as f:
    json.dump({"threshold": float(threshold)}, f, ensure_ascii=False, indent=2)

print(f"Saved model to: {model_path}")
print(f"Saved threshold to: {threshold_path}")

Saved model to: artifacts/catboost_repair_split.cbm
Saved threshold to: artifacts/catboost_repair_split_threshold.json
